# Spark Assignment - Part 2

## Objective

This assignment demonstrates Spark architecture concepts, DataFrame transformations, filtering, schema handling, Spark optimizations, and Parquet operations using PySpark.

In [6]:
!pip install pyspark

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Spark Assignment - Part 2") \
    .getOrCreate()

spark

# Question 1

## Explain the roles of Driver, Cluster Manager, and Executor in a Spark application.

### Answer

### Driver
The Driver is the main process of a Spark application. It creates the SparkSession, converts user code into execution plans, schedules jobs, and coordinates all tasks.

### Cluster Manager
The Cluster Manager allocates CPU and memory resources to Spark applications. It manages the cluster and launches executors on worker nodes. Examples include Standalone, YARN, Mesos, and Kubernetes.

### Executor
Executors are worker processes responsible for executing tasks assigned by the Driver. They perform computations, cache intermediate data, and return results to the Driver.

# Question 2

## How does Spark's Lazy Evaluation strategy improve performance when processing large datasets?

### Answer

Spark uses Lazy Evaluation, meaning transformations are not executed immediately. Instead, Spark records all transformations in a Directed Acyclic Graph (DAG).

Execution begins only when an Action such as show(), count(), or collect() is called.

### Advantages

- Reduces unnecessary computations
- Optimizes execution plans
- Minimizes data movement
- Improves execution speed
- Reduces memory consumption

# Question 3

## Read a CSV file using Spark with Header and Infer Schema enabled.

In [9]:
df = spark.read.csv(
    "/employees.csv",
    header=True,
    inferSchema=True
)

df.show()

+----------+------+----------+---+------+----------+
|EmployeeID|  Name|Department|Age|Salary|Experience|
+----------+------+----------+---+------+----------+
|       101| Rishi|        IT| 22| 50000|         1|
|       102| Rahul|        HR| 28| 42000|         4|
|       103| Priya|   Finance| 31| 65000|         6|
|       104|  Aman|        IT| 25| 55000|         3|
|       105|  Neha|        HR| 27|  NULL|         5|
|       106| Karan|     Sales| 30| 48000|         4|
|       107|Anjali|        IT| 24| 52000|         2|
|       108| Rohit|   Finance| 35| 75000|         8|
+----------+------+----------+---+------+----------+



### Explanation

The read.csv() function loads the employee dataset into a Spark DataFrame.

- header=True uses the first row as column names.
- inferSchema=True automatically detects the data types of each column.

# Question 4

## Difference between CSV and Parquet

### Answer

| CSV | Parquet |
|------|----------|
| Row-based storage | Columnar storage |
| Larger file size | Smaller due to compression |
| Slower reads | Faster reads |
| No schema stored | Schema is stored |
| Better for simple data exchange | Better for analytics and big data |

### Why does it matter?

Parquet reads only the required columns instead of the entire dataset. This reduces disk I/O, improves query performance, and lowers memory usage.

# Question 5

## Select product_id and price where category is 'Electronics'

In [10]:
products = [
    (101, "Laptop", "Electronics", 55000),
    (102, "Mouse", "Electronics", 700),
    (103, "Chair", "Furniture", 2500),
    (104, "Keyboard", "Electronics", 1200),
    (105, "Monitor", "Electronics", 15000)
]

columns = ["product_id", "product_name", "category", "price"]

df_products = spark.createDataFrame(products, columns)

df_products.show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|55000|
|       102|       Mouse|Electronics|  700|
|       103|       Chair|  Furniture| 2500|
|       104|    Keyboard|Electronics| 1200|
|       105|     Monitor|Electronics|15000|
+----------+------------+-----------+-----+



In [11]:
df_products.select("product_id", "price") \
.filter(col("category") == "Electronics") \
.show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101|55000|
|       102|  700|
|       104| 1200|
|       105|15000|
+----------+-----+



# Question 6

## Rename old_name to new_name and cast price from String to Double

In [12]:
products2 = [
    ("Laptop", "55000"),
    ("Mouse", "700"),
    ("Keyboard", "1200")
]

columns = ["old_name", "price"]

df2 = spark.createDataFrame(products2, columns)

df2.show()

+--------+-----+
|old_name|price|
+--------+-----+
|  Laptop|55000|
|   Mouse|  700|
|Keyboard| 1200|
+--------+-----+



In [13]:
df2 = df2.withColumnRenamed("old_name", "new_name")

df2 = df2.withColumn(
    "price",
    col("price").cast("double")
)

df2.printSchema()
df2.show()

root
 |-- new_name: string (nullable = true)
 |-- price: double (nullable = true)

+--------+-------+
|new_name|  price|
+--------+-------+
|  Laptop|55000.0|
|   Mouse|  700.0|
|Keyboard| 1200.0|
+--------+-------+



### Explanation

The column old_name is renamed to new_name and the price column is converted from String to Double using cast().

# Question 7

## How does Spark use the Lineage Graph (DAG) to provide fault tolerance?

### Answer

Spark records every transformation in a Directed Acyclic Graph (DAG), also called the Lineage Graph.

If an Executor or Worker node fails, Spark does not reload the entire dataset. Instead, it recomputes only the lost partitions using the transformations stored in the DAG.

Advantages:

- Automatic fault recovery
- No need for manual checkpoints
- Faster recovery
- Efficient distributed processing

# Question 8

## Filter orders where status is 'Completed' AND amount is greater than 1000.

In [14]:
orders = [
    (1, "Completed", 1200),
    (2, "Pending", 800),
    (3, "Completed", 2500),
    (4, "Cancelled", 1500),
    (5, "Completed", 900)
]

columns = ["order_id", "status", "amount"]

df_orders = spark.createDataFrame(orders, columns)

df_orders.show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|  1200|
|       2|  Pending|   800|
|       3|Completed|  2500|
|       4|Cancelled|  1500|
|       5|Completed|   900|
+--------+---------+------+



In [15]:
df_orders.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
).show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|  1200|
|       3|Completed|  2500|
+--------+---------+------+



### Explanation

The filter() function is used with multiple conditions.

Only orders having status = Completed and amount greater than 1000 are displayed.

# Question 9

## What is Predicate Pushdown and how does it improve Spark performance?

### Answer

Predicate Pushdown is an optimization technique in Spark where filtering conditions are pushed down to the data source before loading the data into memory.

Instead of reading the entire dataset, Spark reads only the required rows.

### Advantages

- Reduces disk I/O
- Reads less data
- Faster execution
- Lower memory usage
- Improves overall performance, especially with Parquet and ORC files

# Question 10

## Add a new column Final_Price after applying a 10% discount.

In [16]:
df_products.show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics|55000|
|       102|       Mouse|Electronics|  700|
|       103|       Chair|  Furniture| 2500|
|       104|    Keyboard|Electronics| 1200|
|       105|     Monitor|Electronics|15000|
+----------+------------+-----------+-----+



In [17]:
df_products = df_products.withColumn(
    "Final_Price",
    col("price") * 0.90
)

df_products.show()

+----------+------------+-----------+-----+-----------+
|product_id|product_name|   category|price|Final_Price|
+----------+------------+-----------+-----+-----------+
|       101|      Laptop|Electronics|55000|    49500.0|
|       102|       Mouse|Electronics|  700|      630.0|
|       103|       Chair|  Furniture| 2500|     2250.0|
|       104|    Keyboard|Electronics| 1200|     1080.0|
|       105|     Monitor|Electronics|15000|    13500.0|
+----------+------------+-----------+-----+-----------+



### Explanation

A new column named Final_Price is created by applying a 10% discount using the withColumn() transformation.

# Question 11

## Difference between Transformations and Actions in Spark

### Answer

| Transformations | Actions |
|-----------------|----------|
| Lazy operations | Trigger execution |
| Create a new DataFrame | Return results |
| Not executed immediately | Executed immediately |
| Examples: filter(), select(), withColumn() | Examples: show(), count(), collect() |


# Question 12

## Save the employee DataFrame in Parquet format.

In [18]:
df.write.mode("overwrite").parquet("/content/employee_parquet")

In [19]:
parquet_df = spark.read.parquet("/content/employee_parquet")

parquet_df.show()

+----------+------+----------+---+------+----------+
|EmployeeID|  Name|Department|Age|Salary|Experience|
+----------+------+----------+---+------+----------+
|       101| Rishi|        IT| 22| 50000|         1|
|       102| Rahul|        HR| 28| 42000|         4|
|       103| Priya|   Finance| 31| 65000|         6|
|       104|  Aman|        IT| 25| 55000|         3|
|       105|  Neha|        HR| 27|  NULL|         5|
|       106| Karan|     Sales| 30| 48000|         4|
|       107|Anjali|        IT| 24| 52000|         2|
|       108| Rohit|   Finance| 35| 75000|         8|
+----------+------+----------+---+------+----------+



### Explanation

The employee dataset is stored in Parquet format and then read again to verify successful storage.

Parquet provides faster query performance and efficient compression.

# Question 13

## Explain why collect() should be avoided on very large datasets.

### Answer

The collect() function transfers all data from Spark Executors to the Driver.

For small datasets, this is acceptable. However, for large datasets it can cause:

- Driver memory overflow
- Slow execution
- Network overhead
- Application crash due to OutOfMemoryError

Instead of collect(), Spark recommends using show(), take(), or limit() for previewing data.

# Question 14

## Display the first five records using show().

In [20]:
df.show(5)

+----------+-----+----------+---+------+----------+
|EmployeeID| Name|Department|Age|Salary|Experience|
+----------+-----+----------+---+------+----------+
|       101|Rishi|        IT| 22| 50000|         1|
|       102|Rahul|        HR| 28| 42000|         4|
|       103|Priya|   Finance| 31| 65000|         6|
|       104| Aman|        IT| 25| 55000|         3|
|       105| Neha|        HR| 27|  NULL|         5|
+----------+-----+----------+---+------+----------+
only showing top 5 rows


# Question 15

## Compare CSV and Parquet storage by writing the same dataset in both formats.

In [21]:
# Save as CSV
df.write.mode("overwrite") \
.option("header", True) \
.csv("/content/output_csv")

# Save as Parquet
df.write.mode("overwrite") \
.parquet("/content/output_parquet")

In [22]:
import os

print("CSV Folder Files:")
print(os.listdir("/content/output_csv"))

print("\nParquet Folder Files:")
print(os.listdir("/content/output_parquet"))

CSV Folder Files:
['.part-00000-91e57874-9fb8-4568-8997-3746f107bb40-c000.csv.crc', 'part-00000-91e57874-9fb8-4568-8997-3746f107bb40-c000.csv', '_SUCCESS', '._SUCCESS.crc']

Parquet Folder Files:
['.part-00000-b09a3e88-6d7e-461a-9c21-71b10ef2bf2d-c000.snappy.parquet.crc', 'part-00000-b09a3e88-6d7e-461a-9c21-71b10ef2bf2d-c000.snappy.parquet', '_SUCCESS', '._SUCCESS.crc']


### Observation

Both CSV and Parquet files are successfully generated.

CSV stores data as plain text, while Parquet stores data in a compressed columnar format.

Parquet generally provides:

- Smaller storage size
- Faster query execution
- Better compression
- Improved analytical performance